# **Esteira de aprendizado de máquina: Classificação de Músicas Virais**

## Etapa 1: escolha do dataset

O dataset escolhido foi "What Makes a Song Viral?", disponível em: https://www.kaggle.com/datasets/vedauv/what-makes-a-song-go-viral-10000-tracks/data?select=viral_songs.csv.
No problema de classificação binária, é preciso identificar se uma música será viral ou não viral (a coluna target é "viral").

In [ ]:
!pip install pandas seaborn matplotlib scikit-learn scipy

In [ ]:
import io
import os
import math
import numpy as np # operações matemáticas
import pandas as pd # permite trabalhar com DataFrame
import seaborn as sns # visualização gráfica
import matplotlib.pyplot as plt # visualização gráfica
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc, confusion_matrix,
    precision_recall_curve, classification_report, f1_score,
)

## Etapa 2: importar o dataset

In [ ]:
df = pd.read_csv('viral_songs.csv')

## Etapa 3: estatísticas descritivas sobre o dataset

In [ ]:
df.shape # quantidade de linhas e colunas (ou seja, quantidade de registros e de atributos)

In [ ]:
df.head(10) # exibe os primeiros 10 registros

In [ ]:
df.columns # nome das colunas

In [ ]:
df.dtypes # tipo de dados das colunas

In [ ]:
df.info() # detalha mais sobre as colunas

In [ ]:
df.describe() # estatísticas sobre todas as colunas, como média, mediana, quartis

In [ ]:
df.describe(include='object')  # estatísticas das colunas de texto (quantidade de valores distintos, mais frequente)

In [ ]:
print(df['viral'].value_counts()) # os valores de labels do problema de classificação
print(df['viral'].value_counts(normalize=True) * 100) # verificar desbalanceamento das classes
df['viral'].value_counts().plot(kind='bar',figsize=(7, 4), rot=0)
plt.title('Distribuição dos valores da target')
plt.xlabel('Classificação')
plt.ylabel('Quantidade de músicas')
plt.show()

## Etapa 4: gráficos de densidade e de caixa

In [ ]:
# gráficos de densidade
colunas_plot = df.select_dtypes(include='number').columns.tolist() # apenas inteiros e decimais
linhas_plot = math.ceil(len(colunas_plot) / 3)
df[colunas_plot].plot(kind='density', subplots=True, layout=(linhas_plot, 3),
                      figsize=(16, 3.2 * linhas_plot), sharex=False)
plt.tight_layout()
plt.show()

In [ ]:
# gráficos de caixa
linhas_plot = math.ceil(len(colunas_plot) / 4)
df[colunas_plot].plot(kind='box', subplots=True, layout=(linhas_plot, 4),
                      figsize=(16, 3 * linhas_plot), sharex=False)
plt.tight_layout()
plt.show()

## Etapa 5: tratamento de dados

### Parte 1: transformação nas linhas

Eliminar linhas de valores ausentes, duplicados ou inválidos. Por exemplo, remover músicas que possuem o mesmo track_id.

In [ ]:
print("Valores ausentes:")
print(df.isnull().sum())
print("Linhas toltamente duplicadas:")
print(df.duplicated().sum())
print("Linhas com track_id repetido:")
print(df['track_id'].duplicated().sum())

In [ ]:
linhas_antes = len(df)
df = df.dropna() # excluir linhas que possuem valores nulos
df = df.drop_duplicates(subset='track_id'); # remover ids duplicados
df = df.drop_duplicates() # excluir linhas duplicadas

regras = {
    'duração inválida': df['duration_sec'] <= 0,
    'bpm inválido': df['tempo_bpm'] <= 0, # bpm negativo
    'viral fora de 0 e 1': ~df['viral'].isin([0,1]) # classificação não é 0 ou 1
}
# criar outras regras para as colunas de escala 0 a 1
colunas_escala_0_a_1 = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'speechiness', 'liveness']
for coluna in colunas_escala_0_a_1:
  if coluna not in df.columns:
    continue
  regras[f"registro de {coluna} fora de [0,1]"] = ~df[coluna].between(0,1)

registros_invalidos = pd.Series(False, index=df.index)
for nome, condicao in regras.items():
    print(f"{nome}: {int(condicao.sum())} linhas")
    registros_invalidos |= condicao

df = df[~registros_invalidos].reset_index(drop=True)

print(f'Quantidade de linhas antes: {linhas_antes}. Linhas restantes {len(df)}')

### Parte 2: transformação nas colunas

1. As colunas track_id e track_name identificam a música, mas não ajudam a prever target.
2. A coluna genre é string e deve ser convertida para um valor numérico. É possível realizar one-hot-enconding em genre porque possui poucos valores diferentes.
3. A coluna artist também é string, porém possui milhares de registros diferentes, impossibilitando one-hot-encondig. Portanto, será removida.
3. A coluna stream é utilizada no dataset para problemas de regressão. No problema de classificação, ela entrega a resposta.

In [ ]:
colunas_remover = ['track_id', 'track_name', 'streams', 'artist']
df = df.drop(columns=[col for col in colunas_remover if col in df.columns])
df['viral'] = df['viral'].astype(int)
df.head()

In [ ]:
# convertendo usando One-Hot-Enconding, já que não queremos correr o risco de adicionar uma "ordem" entre os gêneros.
# cada gênero vira coluna de valores 0 a 1
print(df['genre'].value_counts())
df = pd.get_dummies(df,columns=['genre'], dtype='int8', drop_first=False)
df.head(10)

## Etapa 6: matriz de correlação


Investiga relacionamentos lineares entre variáveis, de 1 a -1: próximo de 1 indica uma relação linear positiva, próxima de -1 uma relação linear negativa, e próxima de 0 uma relação pequena ou inexistente.

In [ ]:
# deixar as colunas de gênero de fora
colunas_numericas = [c for c in df.columns if not c.startswith('genre_')]
matriz_correlacao = df[colunas_numericas].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(matriz_correlacao, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlação entre características musicais')
plt.show()

In [ ]:
# correlação de cada uma com a target
matriz_correlacao['viral'].drop('viral').sort_values()

## Etapa 7: criar subcojuntos de dados (treinamento, testes e validação)

In [ ]:
target_column = 'viral' # define a target
# para usar o algoritmo xgboost, é preciso reordenar de forma que a target seja a primeira coluna
feature_cols = [col for col in df.columns if col != target_column]
df = df[[target_column] + feature_cols]
df.head()

In [ ]:
# separação em 80% para treinamento, 10% para teste e 10% para validação
train, restante = train_test_split(df, test_size=0.2, random_state=42, stratify=df[target_column]) # separa primeiro em 2 grupos, com 20% para test_and_validate
test, validate = train_test_split(restante, test_size=0.5, random_state=42, stratify=restante[target_column]) # separa novamente

In [ ]:
print("treinamento:", train.shape)
print("teste:", test.shape)
print("validação:", validate.shape)

In [ ]:
# verificar proporção entre "viral" e "não viral" em cada divisão:
t1 = train[target_column].value_counts()
t2 = validate[target_column].value_counts()
t3 = test[target_column].value_counts()
distribuicao = pd.concat(
    [t1, t2, t3],
    axis=1
)
distribuicao.columns = ['Treinamento','Validação', 'Teste']
distribuicao.index.name = 'Classificação'
display((distribuicao / distribuicao.sum() * 100).round(2))  # em porcentagem

## Etapa 8: treinamento no algoritmo xgboost

Nessa etapa, enviamos os dados como csv para o S3.

In [ ]:
import boto3
import sagemaker
from sagemaker.image_uris import retrieve
print("Versão sdk do sagemaker: ", sagemaker.__version__)
print("Região da AWS: ", boto3.Session().region_name)

In [ ]:
boto_session = boto3.Session()
s3_resource = boto_session.resource('s3')
s3_client = boto_session.client('s3')
region = boto_session.region_name
sagemaker_session = sagemaker.Session(boto_session=boto_session)
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = "pipeline-viral-songs"

In [ ]:
# função que manda o df para o s3 como .csv, sem cabeçalho ou índice
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
upload_s3_csv('viral_train.csv', 'train', train)
upload_s3_csv('viral_validate.csv', 'validate', validate)

É preciso definir os hiperparâmetros do algoritmo XGBoost:

*   objetive: "binary:logistic": classificação binária pois target vale 0 ou 1.
*   eval_metric: "auc": usa a área sob a curva ROC para validação.
* num_round e early_stopping_rounds: até 200 rodadas, parando quando a métrica AUC como validação não melhora a 20 rodadas.

In [ ]:
nao_virais = int((train[target_column] == 0).sum())
virais = int((train[target_column] == 1).sum())
razao = nao_virais / virais if virais > 0 else 1.0
scale_pos_weight = round(razao, 2) if razao >= 1.5 else 1.0

hyperparams = {
    "objective": "binary:logistic",
    "num_round":"200",
    "eval_metric": "auc",
    "early_stopping_rounds": "20",
    "scale_pos_weight": str(scale_pos_weight)
}

In [ ]:
container = retrieve(framework='xgboost',region=region, version='1.7-1')
s3_output_location = f"s3://{bucket}/{prefix}/output/"

# xgboost
xgb_model = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m4.xlarge',
    output_path=s3_output_location,
    hyperparameters=hyperparams,
    sagemaker_session=sagemaker_session
)

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/train/",
    content_type='text/csv'
)

validate_channel = sagemaker.inputs.TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/validate/",
    content_type='text/csv'
)

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
xgb_model.fit(inputs=data_channels, logs=True)

## Etapa 9: Implamantar modelo

O modelo pode ser usado de duas formas:
* inferência em lote: envia o arquivo com muitas músicas de uma vez e recebe todas as probabilidades. Será usada para os conjuntos de validação e de teste.
* inferência em tempo real: um endpoint ativo que responde com a probabilidade de uma música por vez.


### Batch Transform

In [ ]:
X_validate = validate[feature_cols] # possui apenas as características
y_validate = validate[target_column].to_numpy() # target

X_test = test[feature_cols] # possui apenas as características
y_test = test[target_column].to_numpy() # target

In [ ]:
# enviar validate e test com apenas as características para o S3
upload_s3_csv('viral_validate_features.csv', 'validation-input', X_validate)
upload_s3_csv('viral_test_features.csv','test-input', X_test)

In [ ]:
# função que realiza batch transform no s3 e devolve o caminho com as probabilidades
def batch_predict(input_folder, filename, output_folder):
  transformer = xgb_model.transformer(
    instance_count=1,
    instance_type='ml.m4.xlarge',
    strategy='MultiRecord',
    assemble_with='Line',
    output_path=f's3://{bucket}/{prefix}/{output_folder}/'
  )

  input_path = f's3://{bucket}/{prefix}/{input_folder}/{filename}'

  transformer.transform(
    data=input_path,
    data_type='S3Prefix',
    content_type='text/csv',
    split_type='Line'
  )
  transformer.wait()
  return f'{prefix}/{output_folder}/{filename}.out'

validation_output_key = batch_predict(
    'validation-input',
    'viral_validate_features.csv',
    'validation-output'
)
test_output_key = batch_predict(
    'test-input',
    'viral_test_features.csv',
    'test-output'
)

In [ ]:
# ler arquivo de saída do batch transform
def read_predictions(output_key):
    obj = s3_client.get_object(Bucket=bucket, Key=output_key)
    predictions = pd.read_csv(io.BytesIO(obj['Body'].read()), header=None, names=['viral_predicted'])
    return predictions.iloc[:, 0].astype(float).to_numpy()

# lista de probabilidades de validate e de test
val_prob = read_predictions(validation_output_key) # validação
test_prob = read_predictions(test_output_key) # teste

print("Validação:", len(val_prob), "previsões para", len(y_validate), "músicas")
print("Teste:", len(test_prob), "previsões para", len(y_test), "músicas")

## Etapa 10: avaliação do modelo treinado

### Matriz de Confusão

In [ ]:
# repostas reais
test_labels = pd.Series(y_test, name='viral_real')

# estimativa para cada música, um valor float entre 0 e 1
target_predicted = pd.DataFrame({'viral_predicted': test_prob})

# classificação binária depois de aplicar threshold
target_predicted_binary = pd.DataFrame({
    'viral_predicted': (target_predicted['viral_predicted'] >= 0.5).astype(int)
})

comparacao = pd.DataFrame({
    'real': test_labels,
    'probabilidade': target_predicted['viral_predicted'].round(4),
    'prevista': target_predicted_binary['viral_predicted'],
})
comparacao.head(10)

In [ ]:
matrix = confusion_matrix(test_labels, target_predicted_binary['viral_predicted'], labels=[0,1])
df_confusion = pd.DataFrame(matrix, index=['Not-Viral', 'Viral'],columns=['Not-Viral', 'Viral'])
display(df_confusion)
colormap = sns.color_palette(["#FFC0CB", "#FF1493"])
sns.heatmap(df_confusion, annot=True, cbar=None, cmap=colormap)
plt.title("MATRIZ DE CONFUSÃO")
plt.tight_layout()
plt.ylabel("Verdadeira Classe")
plt.xlabel("Classe Prevista")
plt.show()

### Métricas

In [ ]:
TN, FP, FN, TP = matrix.ravel()

print(f"Verdadeiros negativos (true negatives): {TN}")
print(f"Falsos positivos (false positives): {FP}")
print(f"Falsos negativos (false negatives): {FN}")
print(f"Verdadeiros positivos (true positives) : {TP}")

In [ ]:
# evitar divisão por 0
def razao(numerador, denominador):
    return float(numerador) / denominador * 100 if denominador > 0 else float('nan')

# Acurácia
ACC = razao(TP + TN, TP + TN + FP + FN)
# Sensibilidade
Sensitivity = razao(TP, TP + FN)
# Especificidade (t
Specificity = razao(TN, TN + FP)
# Precisão
Precision = razao(TP, TP + FP)

print(f"Acurácia: {ACC:.2f}% das músicas do teste foram classificadas corretamente.")
print(f"Sensibilidade: {Sensitivity:.2f}% das músicas realmente virais foram detectadas como virais.")
print(f"Especificidade: {Specificity:.2f}% das músicas realmente não virais foram detectadas como não virais.")
print(f"Precisão: quando o modelo diz que a música é viral, ele acerta em {Precision:.2f}% das vezes.")

In [ ]:
print(classification_report(test_labels, target_predicted_binary['viral_predicted'],
                            target_names=['Não viral', 'Viral'], zero_division=0))

## Etapa 11: resultado de uma predição do modelo



O modelo é implantado em um endpoint para inferência em tempo real e recebe as características de uma música do conjunto de teste.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

xgb_predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer()
)

In [ ]:
# primeira música do conjunto de teste; envia somente as características
row = X_test.iloc[[0]]
response = xgb_predictor.predict(row.to_numpy())

probability = float(np.asarray(response).ravel()[0])
classificacao_prevista = int(probability >= 0.5) # aplicar threshold

print("Probabilidade de ser viral:", f"{probability:.2%}")

if classificacao_prevista == 1:
    print("A música foi classificada como VIRAL")
else:
    print("A música foi classificada como NÃO VIRAL")

classificacao_verdadeira = y_test[0]
if classificacao_verdadeira == 1:
    print("A música é viral")
else:
    print("A música é não viral")

if classificacao_verdadeira == classificao_prevista:
  print("O modelo acertou a classificação")
else:
  print("O modelo errou a classificação")

In [ ]:
xgb_predictor.delete_endpoint(delete_endpoint_config=True) # apagar endpoint
xgb_predictor.delete_model()